In [1]:
from pathlib import Path
import pandas as pd
import xarray as xr
import numpy as np
import requests

PROJECT_ROOT = Path(".").resolve()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

OISST_RAW_DIR = RAW_DIR / "oisst"
ONI_RAW_DIR = RAW_DIR / "oni"

for folder in [RAW_DIR, INTERIM_DIR, PROCESSED_DIR, OISST_RAW_DIR, ONI_RAW_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folders created.")

ModuleNotFoundError: No module named 'xarray'

In [ ]:
ONI_URL = (
    "https://www.cpc.ncep.noaa.gov/products/analysis_monitoring/ensostuff/ONI_v5.php"
)

tables = pd.read_html(ONI_URL)

# NOAA 页面通常第一个大表就是 ONI
oni_wide = tables[0]

# 清理列名
oni_wide.columns = [str(c).strip() for c in oni_wide.columns]

oni_wide.head()


In [ ]:
season_to_month = {
    "DJF": 1,
    "JFM": 2,
    "FMA": 3,
    "MAM": 4,
    "AMJ": 5,
    "MJJ": 6,
    "JJA": 7,
    "JAS": 8,
    "ASO": 9,
    "SON": 10,
    "OND": 11,
    "NDJ": 12,
}

oni_long = oni_wide.melt(
    id_vars=["Year"],
    value_vars=list(season_to_month.keys()),
    var_name="season",
    value_name="oni",
)

oni_long["month"] = oni_long["season"].map(season_to_month)
oni_long["date"] = pd.to_datetime(
    dict(year=oni_long["Year"].astype(int), month=oni_long["month"], day=1)
)

oni_long["oni"] = pd.to_numeric(oni_long["oni"], errors="coerce")

oni_long["enso_phase"] = np.select(
    [oni_long["oni"] >= 0.5, oni_long["oni"] <= -0.5],
    ["El Niño", "La Niña"],
    default="Neutral",
)

oni_long = oni_long[["date", "Year", "month", "season", "oni", "enso_phase"]]
oni_long = oni_long.sort_values("date").reset_index(drop=True)

oni_raw_path = ONI_RAW_DIR / "oni_noaa_cpc_raw_wide.csv"
oni_processed_path = PROCESSED_DIR / "oni_monthly.csv"

oni_wide.to_csv(oni_raw_path, index=False)
oni_long.to_csv(oni_processed_path, index=False)

print("Saved raw ONI to:", oni_raw_path)
print("Saved processed ONI to:", oni_processed_path)

oni_long.head()


In [ ]:
OISST_OPENDAP_URL = (
    "https://psl.noaa.gov/thredds/dodsC/"
    "Datasets/noaa.oisst.v2.highres/sst.day.mean.1982.nc"
)

ds_test = xr.open_dataset(OISST_OPENDAP_URL)

ds_test


In [ ]:
# California Coast bounding box
# OISST longitude is 0-360, so:
# -125W = 235E
# -114W = 246E

LAT_MIN = 30
LAT_MAX = 42
LON_MIN = 235
LON_MAX = 246

ds_ca_1982 = ds_test.sel(lat=slice(LAT_MIN, LAT_MAX), lon=slice(LON_MIN, LON_MAX))

ds_ca_1982


In [ ]:
oisst_1982_path = INTERIM_DIR / "california_oisst_1982_test.nc"

ds_ca_1982.to_netcdf(oisst_1982_path)

print("Saved:", oisst_1982_path)


In [ ]:
ca_daily_sst_1982 = (
    ds_ca_1982["sst"].mean(dim=["lat", "lon"], skipna=True).to_dataframe().reset_index()
)

ca_daily_sst_1982 = ca_daily_sst_1982.rename(columns={"time": "date"})
ca_daily_sst_1982["date"] = pd.to_datetime(ca_daily_sst_1982["date"])

ca_daily_sst_1982.head()


In [ ]:
test_daily_path = PROCESSED_DIR / "california_daily_sst_1982_test.parquet"

ca_daily_sst_1982.to_parquet(test_daily_path, index=False)

print("Saved:", test_daily_path)
ca_daily_sst_1982.describe()
